# SatQuery: train the general land-cover model (any change, anywhere) on a free Kaggle GPU

One U-Net, nine land-cover classes (water, trees, grass, flooded vegetation, crops, scrub, built-up, bare, snow/ice),
trained on **Dynamic World** expert labels from every biome, in two versions: multispectral (Sentinel-2 / Landsat)
and true colour (photos, screenshots). SatQuery reads any before/after question as a class change (floods, forest
loss, fire scars, urban growth, drought ...), so no event needs its own model. Tests: Dynamic World's held-out
expert set, and **Sen1Floods11** flood scenes including **Bolivia**, never used for training. A radar water model
(Sen1Floods11) is trained too, for floods under cloud.

**Settings:** Accelerator **GPU T4 x1**, Internet **On**. **Save Version -> Save & Run All**. About 2-3 hours.
Output: `satquery_landcover_models.zip`.

In [ ]:
# 1. Code (generalize branch) and libraries
!nvidia-smi -L
!rm -rf /tmp/SatQuery && git clone -q --depth 1 -b generalize https://github.com/Vansh-Vaid/SatQuery.git /tmp/SatQuery
!cd /tmp/SatQuery && git log --oneline -1
!pip install -q rasterio pystac-client planetary-computer openpyxl 2>&1 | tail -1

In [ ]:
# 2. Dynamic World labels (PANGAEA) + the matching Sentinel-2 scenes (Planetary Computer), about 40-60 min
!cd /tmp/SatQuery && python training/fetch_dynamic_world.py --out /tmp/dynamic_world --train-tiles 1500 --workers 12
!ls /tmp/dynamic_world/train | wc -l; ls /tmp/dynamic_world/test | wc -l

In [ ]:
# 2. Sen1Floods11 hand-labelled data from its public Google Cloud bucket (about 3-4 GB)
import json, os, time, urllib.parse, urllib.request
from concurrent.futures import ThreadPoolExecutor

BUCKET, ROOT = "sen1floods11", "/tmp/sen1floods11"
PREFIXES = ["v1.1/data/flood_events/HandLabeled/S1Hand/", "v1.1/data/flood_events/HandLabeled/S2Hand/",
            "v1.1/data/flood_events/HandLabeled/LabelHand/", "v1.1/splits/flood_handlabeled/"]

def list_objects(prefix):
    names, token = [], None
    while True:
        q = {"prefix": prefix, "fields": "items(name),nextPageToken", "maxResults": 1000}
        if token:
            q["pageToken"] = token
        url = f"https://storage.googleapis.com/storage/v1/b/{BUCKET}/o?" + urllib.parse.urlencode(q)
        d = json.load(urllib.request.urlopen(url, timeout=60))
        names += [i["name"] for i in d.get("items", [])]
        token = d.get("nextPageToken")
        if not token:
            return names

def fetch(name):
    dst = os.path.join(ROOT, name)
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        return
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    url = f"https://storage.googleapis.com/{BUCKET}/" + urllib.parse.quote(name)
    for attempt in range(4):
        try:
            urllib.request.urlretrieve(url, dst + ".part")
            os.replace(dst + ".part", dst)
            return
        except Exception:
            time.sleep(3 * (attempt + 1))
    raise RuntimeError(f"could not download {name}")

t0 = time.time()
names = [n for p in PREFIXES for n in list_objects(p) if n.endswith((".tif", ".csv"))]
print({p.split("/")[-2]: sum(n.startswith(p) for n in names) for p in PREFIXES})
if not names:
    top = json.load(urllib.request.urlopen(f"https://storage.googleapis.com/storage/v1/b/{BUCKET}/o?delimiter=/&prefix=v1.1/", timeout=60))
    raise SystemExit(f"nothing found; top-level folders are: {top.get('prefixes')}")
with ThreadPoolExecutor(16) as ex:
    list(ex.map(fetch, names))
print(f"{len(names)} files in {(time.time() - t0) / 60:.1f} min")

In [ ]:
# 4. General land-cover model (multispectral + true colour); scored on held-out experts and on flood scenes
!cd /tmp/SatQuery && python training/train_landcover.py --data /tmp/dynamic_world --out /kaggle/working/landcover_seg.pt \
    --sen1floods11 /tmp/sen1floods11 --epochs 40

In [ ]:
# 5. Radar water model (Sentinel-1), for floods under cloud
!cd /tmp/SatQuery && python training/train_flood.py --root /tmp/sen1floods11 --out /kaggle/working/flood_seg.pt --variants s1 --epochs 60

In [ ]:
# 6. Verdicts and packing
import json, zipfile
lc = json.load(open("/kaggle/working/landcover_seg.pt.metrics.json"))
for v, r in lc["variants"].items():
    print(f"{v:4s} 9-class mIoU {r['test_miou']:.3f} | 4-class accuracy {r['test_acc_4class']:.3f} vs rules "
          f"{r['baseline_acc_4class']:.3f} | flood scenes water IoU {r.get('sen1floods11_test_water_iou')} "
          f"(Bolivia {r.get('sen1floods11_bolivia_water_iou')}) -> {'BETTER' if r['beats_baseline'] else 'not better'}")
fl = json.load(open("/kaggle/working/flood_seg.pt.metrics.json"))["variants"]["s1"]
print(f"radar water IoU {fl['test_iou']:.3f} vs Otsu {fl['baseline_iou']:.3f} (Bolivia {fl.get('bolivia_iou')}) -> "
      f"{'BETTER' if fl['beats_baseline'] else 'not better'}")
with zipfile.ZipFile("/kaggle/working/satquery_landcover_models.zip", "w") as z:
    for f in ("landcover_seg.pt", "landcover_seg.pt.metrics.json", "flood_seg.pt", "flood_seg.pt.metrics.json"):
        z.write(f"/kaggle/working/{f}", f)
print("Download satquery_landcover_models.zip from the Output tab.")

## Afterwards
Unzip into `checkpoints/`. `change_mapper` uses each version only if it is marked BETTER.